# Assignment 7.1: Data Preprocessing

**Name:** Umang Patel  
**Course:** AIT 620: Methods of Artificial Intelligence and Machine Learning  
**Professor:** Dr. Fuat Akal  
**Date:** October 15, 2026

In this assignment I clean the United Nations data on immigration to Canada from 1980 to 2013. When the Excel file is first read into a DataFrame, it has extra code columns that are not useful, column names that are hard to read, and the countries are just numbered rows. My goal is to turn it into a clean table with one row per country, easy column names, and a total for each country.

## Step 1: Import the libraries

In [1]:
import pandas as pd
import numpy as np

## Step 2: Read the data into a DataFrame
The data is in the sheet called **Canada by Citizenship**. The first 20 rows of that sheet are only title information from the UN, and the last 2 rows are **Unknown** and **Total**, which are not real countries. So I skip those rows while reading the file.

In [2]:
# names of the sheets in the Excel file
print(pd.ExcelFile('Canada.xlsx').sheet_names)

['Regions by Citizenship', 'Canada by Citizenship']


In [3]:
# skip the 20 title rows at the top and the 2 total rows at the bottom
df_can = pd.read_excel('Canada.xlsx',
                       sheet_name='Canada by Citizenship',
                       skiprows=range(20),
                       skipfooter=2)

print(df_can.shape)
df_can.head()

(195, 43)


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,0,1,1,0,0,0,0,1,1


Now we have 195 countries and 43 columns. Let me check the data types and see if anything is missing.

In [4]:
# column names, data types and missing values
df_can.info(verbose=False)
print()
print("Missing values:", df_can.isnull().sum().sum())

<class 'pandas.DataFrame'>
RangeIndex: 195 entries, 0 to 194
Columns: 43 entries, Type to 2013
dtypes: int64(37), str(6)
memory usage: 65.6 KB

Missing values: 0


There are no missing values, so nothing needs to be filled in.

## Step 3: Remove columns we do not need
Some columns are not useful for analysis:

- **AREA, REG, DEV** are just number codes for the continent, region, and development level. We already have the names in AreaName, RegName, and DevName.
- **Type** and **Coverage** are almost the same for every row.

In [5]:
# Type and Coverage barely change, so they are not useful
print(df_can['Type'].value_counts())
print()
print(df_can['Coverage'].value_counts())

Type
Immigrants    195
Name: count, dtype: int64

Coverage
Foreigners    194
Citizens        1
Name: count, dtype: int64


In [6]:
# drop the columns we do not need
df_can.drop(['AREA', 'REG', 'DEV', 'Type', 'Coverage'], axis=1, inplace=True)
df_can.head()

,OdName,AreaName,RegName,DevName,1980,1981,1982,1983,1984,1985,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,...,1450,1223,856,702,560,716,561,539,620,603
2,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
4,Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,1,1


## Step 4: Rename columns
The names **OdName**, **AreaName**, and **RegName** are not clear, so I give them simpler names.

In [7]:
# give the columns easier names
df_can.rename(columns={'OdName': 'Country', 'AreaName': 'Continent', 'RegName': 'Region'}, inplace=True)
df_can.columns.tolist()

['Country',
 'Continent',
 'Region',
 'DevName',
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013]

## Step 5: Make all column names text
The year columns are numbers (1980, 1981, ...) while the other column names are text. Mixing the two can cause problems later, so I change all column names to text.

In [8]:
# check the types of the column names
print(all(isinstance(column, str) for column in df_can.columns))

# change all column names to strings
df_can.columns = list(map(str, df_can.columns))

print(all(isinstance(column, str) for column in df_can.columns))

False
True


## Step 6: Use the country name as the index
This makes it easy to look up a country, for example `df_can.loc['India']`.

In [9]:
# set Country as the index
df_can.set_index('Country', inplace=True)

df_can.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1450,1223,856,702,560,716,561,539,620,603
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,0,1,1,0,0,0,0,1,1


## Step 7: Add a Total column
I add up the immigrants for every year from 1980 to 2013 to get the total for each country.

In [10]:
# list of year columns as text
years = list(map(str, range(1980, 2014)))

# total immigrants for each country
df_can['Total'] = df_can[years].sum(axis=1)
df_can.head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15


## Step 8: Check the final result

In [11]:
# size of the clean table
print(df_can.shape)

(195, 38)


In [12]:
# summary of the clean table
df_can.info(verbose=False)

<class 'pandas.DataFrame'>
Index: 195 entries, Afghanistan to Zimbabwe
Columns: 38 entries, Continent to Total
dtypes: int64(35), str(3)
memory usage: 59.4+ KB


In [13]:
# example: look up one country
df_can.loc[['India', 'China', 'Philippines']]

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
India,Asia,Southern Asia,Developing regions,8880,8670,8147,7338,5704,4211,7150,...,36210,33848,28742,28261,29456,34235,27509,30933,33087,691904
China,Asia,Eastern Asia,Developing regions,5123,6682,3308,1863,1527,1816,1960,...,42584,33518,27642,30037,29622,30391,28502,33024,34129,659962
Philippines,Asia,South-Eastern Asia,Developing regions,6051,5921,5249,4562,3801,3150,4166,...,18139,18400,19837,24887,28573,38617,36765,34315,29544,511391


In [14]:
# the 5 countries with the most immigrants to Canada
df_can.sort_values('Total', ascending=False).head()

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
India,Asia,Southern Asia,Developing regions,8880,8670,8147,7338,5704,4211,7150,...,36210,33848,28742,28261,29456,34235,27509,30933,33087,691904
China,Asia,Eastern Asia,Developing regions,5123,6682,3308,1863,1527,1816,1960,...,42584,33518,27642,30037,29622,30391,28502,33024,34129,659962
United Kingdom of Great Britain and Northern Ireland,Europe,Northern Europe,Developed regions,22045,24796,20620,10015,10170,9564,9470,...,7258,7140,8216,8979,8876,8724,6204,6195,5827,551500
Philippines,Asia,South-Eastern Asia,Developing regions,6051,5921,5249,4562,3801,3150,4166,...,18139,18400,19837,24887,28573,38617,36765,34315,29544,511391
Pakistan,Asia,Southern Asia,Developing regions,978,972,1201,900,668,514,691,...,14314,13127,10124,8994,7217,6811,7468,11227,12603,241600


## Final DataFrame

In [15]:
df_can

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Viet Nam,Asia,South-Eastern Asia,Developing regions,1191,1829,2162,3404,7583,5907,2741,...,1852,3153,2574,1784,2171,1942,1723,1731,2112,97146
Western Sahara,Africa,Northern Africa,Developing regions,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,2
Yemen,Asia,Western Asia,Developing regions,1,2,1,6,0,18,7,...,161,140,122,133,128,211,160,174,217,2985


## Summary

After reading the data, the table had 195 countries but also extra code columns and names that were hard to read. To clean it, I:

1. Removed the code columns (AREA, REG, DEV) and the columns that barely change (Type, Coverage)
2. Renamed OdName, AreaName and RegName to Country, Continent and Region
3. Changed all column names to text
4. Set Country as the index
5. Added a Total column for 1980 to 2013

The final table has **195 countries** and **38 columns** (Continent, Region, DevName, 34 years, and Total), and it is ready to be used for analysis and charts. For example, it shows that India, China and the United Kingdom sent the most immigrants to Canada over this time.